# 🧩 Mini-Lab: Reference-Based Metrics

**Module 9: LLM Evaluation** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why systematic evaluation with reference-based metrics matters for LLM applications
2. **Compute** BLEU scores to measure n-gram overlap between generated and reference text
3. **Compute** ROUGE scores to measure recall-oriented overlap for summarization tasks
4. **Compare** BLEU and ROUGE to choose the right metric for different use cases

## Target Concepts

| Concept | Description |
|---------|-------------|
| Why Evaluation Matters | Importance of systematic testing — "vibes-based" checking doesn't scale |
| Reference-Based Evaluations | BLEU, ROUGE, and exact match — objective metrics that compare generated text against a known-good reference |

## Why Reference-Based Evaluation?

When building LLM applications, you need to know whether your outputs are *good*. Manually reading every response doesn't scale. Reference-based metrics solve this by **comparing generated text against a known-good reference answer** and producing a numeric score.

Two of the most widely used reference-based metrics are:

| Metric | Measures | Best For |
|--------|----------|----------|
| **BLEU** | Precision — how much of the *generated* text appears in the reference | Translation, short factual answers |
| **ROUGE** | Recall — how much of the *reference* text is captured in the generation | Summarization, long-form answers |

## Setup

In [2]:
import nltk
nltk.download('punkt_tab', quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Sample Evaluation Data

Let's create a small evaluation set — three question-answer pairs with a **reference answer** (the gold standard) and a **generated answer** (what an LLM might produce).

In [3]:
eval_data = [
    {
        "question": "What is the capital of France?",
        "reference": "The capital of France is Paris.",
        "generated": "Paris is the capital of France."
    },
    {
        "question": "Summarize the benefits of exercise.",
        "reference": "Exercise improves cardiovascular health, boosts mood, strengthens muscles, and helps maintain a healthy weight.",
        "generated": "Working out boosts mood and strengthens muscles."
    },
    {
        "question": "What does RAG stand for?",
        "reference": "RAG stands for Retrieval-Augmented Generation.",
        "generated": "RAG means Retrieval-Augmented Generation, a technique for grounding LLM responses."
    }
]

## BLEU Score

**BLEU** (Bilingual Evaluation Understudy) measures **precision**: what fraction of n-grams in the generated text also appear in the reference?

- A score of **1.0** means perfect overlap
- A score of **0.0** means no overlap

We use smoothing to avoid zero scores when some n-gram orders have no matches.

In [4]:
smoother = SmoothingFunction().method1

results = []
for item in eval_data:
    reference_tokens = item["reference"].lower().split()
    generated_tokens = item["generated"].lower().split()
    
    score = sentence_bleu(
        [reference_tokens],
        generated_tokens,
        smoothing_function=smoother
    )
    results.append(score)

# Display results
lines = ["| Question | BLEU Score |", "|----------|------------|"]
for item, score in zip(eval_data, results):
    lines.append(f"| {item['question']} | **{score:.3f}** |")

avg_bleu = sum(results) / len(results)
lines.append(f"\n**Average BLEU: {avg_bleu:.3f}**")
md("\n".join(lines))

| Question | BLEU Score |
|----------|------------|
| What is the capital of France? | **0.217** |
| Summarize the benefits of exercise. | **0.016** |
| What does RAG stand for? | **0.028** |

**Average BLEU: 0.087**

Notice how the first example scores high — the words are nearly identical, just reordered. The second example scores low because the generated answer uses different words ("working out" vs. "exercise") and misses key content.

## ROUGE Score

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures **recall**: how much of the reference text is captured in the generated text?

Common variants:
- **ROUGE-1**: unigram (single word) overlap
- **ROUGE-2**: bigram (two-word) overlap
- **ROUGE-L**: longest common subsequence

In [5]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

lines = [
    "| Question | ROUGE-1 (F1) | ROUGE-2 (F1) | ROUGE-L (F1) |",
    "|----------|-------------|-------------|-------------|"
]

all_scores = []
for item in eval_data:
    scores = scorer.score(item["reference"], item["generated"])
    all_scores.append(scores)
    lines.append(
        f"| {item['question']} "
        f"| **{scores['rouge1'].fmeasure:.3f}** "
        f"| **{scores['rouge2'].fmeasure:.3f}** "
        f"| **{scores['rougeL'].fmeasure:.3f}** |"
    )

# Averages
avg_r1 = sum(s['rouge1'].fmeasure for s in all_scores) / len(all_scores)
avg_r2 = sum(s['rouge2'].fmeasure for s in all_scores) / len(all_scores)
avg_rl = sum(s['rougeL'].fmeasure for s in all_scores) / len(all_scores)
lines.append(f"\n**Averages — ROUGE-1: {avg_r1:.3f} | ROUGE-2: {avg_r2:.3f} | ROUGE-L: {avg_rl:.3f}**")

md("\n".join(lines))

| Question | ROUGE-1 (F1) | ROUGE-2 (F1) | ROUGE-L (F1) |
|----------|-------------|-------------|-------------|
| What is the capital of France? | **1.000** | **0.600** | **0.667** |
| Summarize the benefits of exercise. | **0.476** | **0.211** | **0.381** |
| What does RAG stand for? | **0.588** | **0.267** | **0.471** |

**Averages — ROUGE-1: 0.688 | ROUGE-2: 0.359 | ROUGE-L: 0.506**

## ROUGE: Precision vs. Recall vs. F1

ROUGE provides three sub-scores for each variant. Let's look at example 2 (the exercise summary) to understand the difference.

In [6]:
item = eval_data[1]
scores = scorer.score(item["reference"], item["generated"])

lines = [
    f"**Reference:** {item['reference']}\n",
    f"**Generated:** {item['generated']}\n",
    "",
    "| Metric | Precision | Recall | F1 |",
    "|--------|-----------|--------|----|",
    f"| ROUGE-1 | {scores['rouge1'].precision:.3f} | {scores['rouge1'].recall:.3f} | {scores['rouge1'].fmeasure:.3f} |",
    f"| ROUGE-2 | {scores['rouge2'].precision:.3f} | {scores['rouge2'].recall:.3f} | {scores['rouge2'].fmeasure:.3f} |",
    f"| ROUGE-L | {scores['rougeL'].precision:.3f} | {scores['rougeL'].recall:.3f} | {scores['rougeL'].fmeasure:.3f} |",
    "",
    "- **Precision**: Of the words the model generated, how many were in the reference?",
    "- **Recall**: Of the words in the reference, how many did the model capture?",
    "- **F1**: The harmonic mean — balances both"
]

md("\n".join(lines))

**Reference:** Exercise improves cardiovascular health, boosts mood, strengthens muscles, and helps maintain a healthy weight.

**Generated:** Working out boosts mood and strengthens muscles.


| Metric | Precision | Recall | F1 |
|--------|-----------|--------|----|
| ROUGE-1 | 0.714 | 0.357 | 0.476 |
| ROUGE-2 | 0.333 | 0.154 | 0.211 |
| ROUGE-L | 0.571 | 0.286 | 0.381 |

- **Precision**: Of the words the model generated, how many were in the reference?
- **Recall**: Of the words in the reference, how many did the model capture?
- **F1**: The harmonic mean — balances both

Notice the recall is low — the generated answer only captured part of the reference. The precision is higher because the words it *did* use were relevant. This is exactly the kind of insight that helps you decide whether your LLM is being too brief or too verbose.

## Exact Match

The simplest reference-based metric: does the generated answer **exactly** match the reference? Useful for factual lookups and structured outputs.

In [7]:
def exact_match(reference, generated):
    """Case-insensitive exact match."""
    return reference.strip().lower() == generated.strip().lower()

lines = ["| Question | Exact Match |", "|----------|-------------|"]
for item in eval_data:
    match = exact_match(item["reference"], item["generated"])
    lines.append(f"| {item['question']} | {'✅' if match else '❌'} |")

md("\n".join(lines))

| Question | Exact Match |
|----------|-------------|
| What is the capital of France? | ❌ |
| Summarize the benefits of exercise. | ❌ |
| What does RAG stand for? | ❌ |

Exact match is strict — even "Paris is the capital of France" vs. "The capital of France is Paris" fails. That's why BLEU and ROUGE are more useful for free-form text.

## Comparing All Metrics Side by Side

In [8]:
lines = [
    "| # | Question | Exact Match | BLEU | ROUGE-1 (F1) | ROUGE-L (F1) |",
    "|---|----------|-------------|------|-------------|-------------|"
]

smoother = SmoothingFunction().method1
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

for i, item in enumerate(eval_data, 1):
    em = "✅" if exact_match(item["reference"], item["generated"]) else "❌"
    
    bleu = sentence_bleu(
        [item["reference"].lower().split()],
        item["generated"].lower().split(),
        smoothing_function=smoother
    )
    
    rouge = scorer.score(item["reference"], item["generated"])
    
    lines.append(
        f"| {i} | {item['question']} | {em} "
        f"| {bleu:.3f} | {rouge['rouge1'].fmeasure:.3f} "
        f"| {rouge['rougeL'].fmeasure:.3f} |"
    )

lines.append("")
lines.append("**When to use which:**")
lines.append("- **Exact Match** → structured outputs, short factual lookups")
lines.append("- **BLEU** → translation, precise wording matters")
lines.append("- **ROUGE** → summarization, capturing key information matters")

md("\n".join(lines))

| # | Question | Exact Match | BLEU | ROUGE-1 (F1) | ROUGE-L (F1) |
|---|----------|-------------|------|-------------|-------------|
| 1 | What is the capital of France? | ❌ | 0.217 | 1.000 | 0.667 |
| 2 | Summarize the benefits of exercise. | ❌ | 0.016 | 0.476 | 0.381 |
| 3 | What does RAG stand for? | ❌ | 0.028 | 0.588 | 0.471 |

**When to use which:**
- **Exact Match** → structured outputs, short factual lookups
- **BLEU** → translation, precise wording matters
- **ROUGE** → summarization, capturing key information matters

## 🎯 Summary

### Key Takeaways

1. **Systematic evaluation matters** — manually checking LLM outputs doesn't scale; numeric metrics let you track quality automatically across hundreds of examples
2. **BLEU measures precision** — it checks how much of the generated text overlaps with the reference, making it suitable for translation and factual answers
3. **ROUGE measures recall** — it checks how much of the reference is captured in the generation, making it ideal for summarization tasks
4. **Exact match is strict but useful** — perfect for structured outputs where wording must be precise
5. **No single metric is enough** — BLEU, ROUGE, and exact match each capture different aspects of quality; choose based on your task